# 30 · Pig instance segmentation — visual error audit
### FaroPigSeg · First research article · Descriptive evaluation on previously inspected TEST images

**Runtime:** **CPU (no GPU required)**. The original experiment 23 contains the segmentation masks (COCO RLE) for all three YOLO26 seeds. Reusing them is preferable to running fresh inference because the prediction provenance is preserved exactly.

**Research questions:** Which predicted masks disappear after validation-calibrated confidence thresholds? Are missed animals, overlapping instances, or segmentation mismatches visually associated with the residual counting errors? These are *descriptive*, hypothesis-generating analyses rather than independent validation or causal explanations.

**Pre-existing model outputs:** 2026: 0.70 · 2027: 0.65 · 2028: 0.65. Baseline threshold: 0.25. **No threshold tuning on test data.**

**Inputs in Google Drive**
- `05_Exportaciones_ZIP/CERDOS_23_YOLO26_Evaluacion_Resultados.zip` (original predictions, masks and evaluation table)
- `03_Resultados/CERDOS_29_Analisis_Casos_Criticos/CERDOS_29_Analisis_Casos_Criticos.xlsx`
- `01_Datasets/FaroPigSeg/extraido/FaroPigSeg/{train,val,test}/{images,labels}` (original storage; experimental split comes from the **date manifest**, not from original folders)
- `01_Datasets/FaroPigSeg/split_fecha_v1/particion_fecha_manifest.json`

**Critical path correction:** experimental TEST images are physically stored in the dataset’s original train, val, and test folders. Never equate an original folder name with the project date-based experimental split.

**Outputs:** `03_Resultados/CERDOS_30_Analisis_Visual_Errores`, Excel workbook, 600-DPI English figures, PDF/SVG, a provenance JSON and downloadable ZIP. The original files and notebooks are never overwritten.

## 1. Environment and paths
The notebook reads from existing Drive folders and checks every required file before processing. It intentionally does **not** install Ultralytics or open model weights, because COCO masks from experiment 23 are already available. Run **Runtime → Change runtime type → CPU**.

**Input correction:** search all three original FaroPigSeg storage folders and require one unambiguous image/label pair; test-set membership comes exclusively from `particion_fecha_manifest.json`.


In [ ]:
%pip -q install openpyxl
from google.colab import drive
from pathlib import Path
import pandas as pd, numpy as np, matplotlib, matplotlib.pyplot as plt
from PIL import Image
import json, hashlib, zipfile, io, cv2, re, platform, sys, importlib.metadata
from collections import defaultdict
from matplotlib.patches import Patch
from IPython.display import display

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
ZIP23 = ROOT / '05_Exportaciones_ZIP/CERDOS_23_YOLO26_Evaluacion_Resultados.zip'
XLSX29 = ROOT / '03_Resultados/CERDOS_29_Analisis_Casos_Criticos/CERDOS_29_Analisis_Casos_Criticos.xlsx'
DATASET = ROOT / '01_Datasets/FaroPigSeg/extraido/FaroPigSeg'
ORIGINAL_STORAGE_SPLITS = ('train', 'val', 'test')
SOURCE_PAIRS = {split: (DATASET / split / 'images', DATASET / split / 'labels') for split in ORIGINAL_STORAGE_SPLITS}
MANIFEST = ROOT / '01_Datasets/FaroPigSeg/split_fecha_v1/particion_fecha_manifest.json'
OUT = ROOT / '03_Resultados/CERDOS_30_Analisis_Visual_Errores'
FIG_DIR = OUT / 'Figures'
# Compare original frozen thresholds; NEVER optimize on test.
BASELINE_CONF = 0.25
FROZEN = {2026: 0.70, 2027: 0.65, 2028: 0.65}
MATCH_IOU = 0.50                  # only for descriptive TP/FP/FN matching; NOT COCO mAP
MAX_EXTRA_RESIDUAL = 6
MAX_IMPROVED_CONTROLS = 3
MAX_SEVEN_PANEL_FIGURES = 6
FIG_DPI = 600
for path in [ZIP23, XLSX29, DATASET, MANIFEST] + [folder for pair in SOURCE_PAIRS.values() for folder in pair]:
    print(('FOUND  ' if path.exists() else 'MISSING '), str(path))
missing = [str(p) for p in [ZIP23, XLSX29, DATASET, MANIFEST] + [folder for pair in SOURCE_PAIRS.values() for folder in pair] if not p.exists()]
if missing:
    raise FileNotFoundError('Required project files are missing from your mounted Drive:\n' + '\n'.join(missing))
OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':10, 'axes.labelsize':11, 'axes.titlesize':11,
    'axes.spines.top':False, 'axes.spines.right':False, 'pdf.fonttype':42, 'svg.fonttype':'none',
    'savefig.facecolor':'white', 'figure.facecolor':'white'})
print('CPU notebook ready. Original train/val/test are STORAGE folders, not the experiment date split.')
print('Date-manifest TEST membership will be validated independently.')

## 2. Load and validate experiment provenance
Cross-check the dated TEST split, test image IDs, all seed counts and frozen thresholds against independent experiment 29 records. Execution stops instead of silently using the wrong file or different predictions.

In [ ]:
assert BASELINE_CONF == 0.25 and FROZEN == {2026:0.70,2027:0.65,2028:0.65}
with open(MANIFEST, encoding='utf-8') as f: manifest = json.load(f)
date_map = manifest['asignaciones_fecha']
test_dates = {date for date,split in date_map.items() if split == 'test'}
assert len(test_dates) == 5, f'Unexpected test dates: {test_dates}'
with zipfile.ZipFile(ZIP23) as z:
    needed=['CERDOS_23_YOLO26_Evaluacion_COCO_3_Semillas.xlsx','CERDOS_23_Protocolo.json']+[f'CERDOS_23_Pred_seed_{s}.json' for s in FROZEN]
    assert all(k in z.namelist() for k in needed), 'Incomplete experiment 23 ZIP'
    protocol=json.loads(z.read('CERDOS_23_Protocolo.json'))
    table23=pd.read_excel(io.BytesIO(z.read(needed[0])), sheet_name='Conteo_Por_Imagen')
    raw_pred={seed:json.loads(z.read(f'CERDOS_23_Pred_seed_{seed}.json')) for seed in FROZEN}
assert protocol['prediction']['conf'] <= 0.00101 and protocol['prediction']['iou'] == 0.7
assert protocol['prediction']['retina_masks'] is True
assert protocol['count_conf_threshold'] == BASELINE_CONF
book29=pd.read_excel(XLSX29, sheet_name=None)
assert {'All_Seeds_Worsened','Image_Review_Priority','Per_Image_Detail','Frozen_Thresholds'} <= set(book29), 'Unexpected experiment 29 workbook'
critical=book29['All_Seeds_Worsened'].copy()
priority=book29['Image_Review_Priority'].copy()
per29=book29['Per_Image_Detail'].copy()
locked=book29['Frozen_Thresholds']
assert len(critical)==21 and len(priority)==373 and len(per29)==1119
assert {int(x.seed):float(x.threshold) for x in locked.itertuples(index=False)} == FROZEN
assert len(set(priority['imagen']))==373 and len(set(priority['image_id']))==373
assert set(priority['fecha']).issubset(test_dates)
assert all(date_map[x]=='test' for x in priority['fecha'])
assert priority['imagen'].apply(lambda s: re.match(r'^\d{4}-\d{2}-\d{2}', str(s)) is not None).all()
assert (priority['imagen'].str[:10] == priority['fecha']).all(), 'Image dates do not match manifest dates'
assert all((priority['image_id'].astype(int) == priority['image_id']).tolist())
assert len(table23)==1119
# All 3 seeds must reference exactly the same test images.
for seed in FROZEN:
    x=table23[table23['semilla']==seed]
    assert len(x)==373 and set(x['image_id'])==set(priority['image_id'])
    merged=x[['image_id','imagen','gt_conteo']].merge(priority[['image_id','imagen','gt_conteo']], on='image_id', suffixes=('_23','_29'),validate='one_to_one')
    assert (merged['imagen_23']==merged['imagen_29']).all() and (merged['gt_conteo_23']==merged['gt_conteo_29']).all()
print('Validated: 373 TEST images / 21 jointly worsened / 3 seeds / 5 dates.')
print('Prediction protocol:',protocol['prediction'])

## 3. Reproduce counts using the original RLE predictions
No model inference, changed NMS, or new confidence threshold selection. Both baseline and calibrated counts must **exactly** match the previous notebooks for every image and seed.

In [ ]:
byseed={}
counts=[]
for seed, pred in raw_pred.items():
    index=defaultdict(list)
    for det in pred:
        index[int(det['image_id'])].append(det)
    byseed[seed]=dict(index)
    r=per29[per29['semilla']==seed].set_index('image_id')
    q=table23[table23['semilla']==seed].set_index('image_id')
    for image_id in priority['image_id'].astype(int):
        det=index.get(image_id, [])
        n0=sum(float(d['score'])>=BASELINE_CONF for d in det)
        n1=sum(float(d['score'])>=FROZEN[seed] for d in det)
        expected0=int(r.loc[image_id,'count_original'])
        expected1=int(r.loc[image_id,'count_calibrated'])
        counts.append(dict(seed=seed,image_id=image_id,baseline=n0,calibrated=n1,baseline_recorded=expected0,
                           calibrated_recorded=expected1,match=(n0==expected0 and n1==expected1)))
countcheck=pd.DataFrame(counts)
if not countcheck['match'].all():
    display(countcheck[~countcheck['match']].head(12))
    raise AssertionError('Experiment 23 COCO masks do not reproduce experiment 29 counts. Stopping to prevent incorrect visual analysis.')
print('PASS — all 1,119 original and calibrated image-seed counts reproduced exactly from saved RLE predictions.')

## 4. Select representative cases and verify image/label alignment
Every one of the 21 jointly worsened cases is included. A small group of high-residual and improved controls helps avoid presenting only failures. Samples are selected for qualitative inspection, **not** threshold tuning.

In [ ]:
critical_ids=critical['image_id'].astype(int).tolist()
extras=priority[(~priority['image_id'].isin(critical_ids))].sort_values(['worst_residual_pigs','n_seeds_large_residual','image_id'], ascending=[False,False,True]).head(MAX_EXTRA_RESIDUAL)['image_id'].astype(int).tolist()
controls=priority[(~priority['image_id'].isin(critical_ids+extras)) & (priority['all_seeds_improved']==True)].sort_values('image_id').head(MAX_IMPROVED_CONTROLS)['image_id'].astype(int).tolist()
selected=critical_ids+extras+controls
assert len(selected)==len(set(selected))
record=priority.set_index('image_id')
# Build a filename index across ALL original FaroPigSeg storage folders.
# The project uses a different TEST split by acquisition date, not by physical directory name.
# Never fall back to a similarly named file or silently select the first match.
source_index=defaultdict(list)
for original_split,(images_folder,labels_folder) in SOURCE_PAIRS.items():
    for image_file in images_folder.iterdir():
        if image_file.is_file() and image_file.suffix.lower() in ('.jpeg','.jpg','.png'):
            label_file=labels_folder/(image_file.stem+'.txt')
            source_index[image_file.name].append((original_split,image_file,label_file))
case_rows=[]
for image_id in selected:
    row=record.loc[image_id]
    matches=source_index.get(str(row['imagen']), [])
    if len(matches)!=1:
        raise FileNotFoundError(f'Image ID {image_id}: expected exactly one original FaroPigSeg file for {row["imagen"]!r}; found {len(matches)} across train/val/test')
    storage_split,img,lab=matches[0]
    if not img.is_file() or not lab.is_file():
        raise FileNotFoundError(f'Image ID {image_id}: missing image or original YOLO polygon label: {img} | {lab}')
    group='All three seeds worsened' if image_id in critical_ids else ('Large residual' if image_id in extras else 'Improved control')
    case_rows.append({'image_id':image_id,'date':str(row['fecha']),'filename':str(row['imagen']),
                      'group':group,'gt_count':int(row['gt_conteo']), 'original_storage_folder':storage_split, 'image_path':str(img),
                      'label_path':str(lab),'image_exists':img.is_file(),'label_exists':lab.is_file()})
selected_cases=pd.DataFrame(case_rows)
if not (selected_cases['image_exists'] & selected_cases['label_exists']).all():
    display(selected_cases.loc[~(selected_cases['image_exists'] & selected_cases['label_exists'])])
    raise FileNotFoundError('Selected images or polygon labels are missing. Confirm the original FaroPigSeg dataset is extracted without renaming.')
print('Selected',len(selected_cases),'images:',len(critical_ids),'jointly worsened;',len(extras),'residual;',len(controls),'controls.')
print('Original storage locations (not experimental partitions):',selected_cases['original_storage_folder'].value_counts().to_dict())
display(selected_cases[['image_id','date','group','gt_count']].head(12))

## 5. Decode instance masks and perform descriptive IoU matching
The IoU≥0.50 one-to-one matching is used only for an **error taxonomy** on selected images. It is *not* equivalent to COCO evaluation. Low-IoU instances are counted as unmatched for this descriptive inspection; visual review is required before assigning a cause such as crowding or occlusion.

In [ ]:
def load_reference(image_path,label_path):
    im=cv2.imread(str(image_path))
    if im is None: raise ValueError(f'Unreadable image: {image_path}')
    rgb=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
    h,w=rgb.shape[:2]
    masks=[]
    for lineno,line in enumerate(label_path.read_text(encoding='utf-8').splitlines(),start=1):
        f=line.strip().split()
        if not f: continue
        if len(f)<7 or (len(f)-1)%2: raise ValueError(f'Invalid YOLO polygon in {label_path.name}:{lineno}')
        coords=np.array(list(map(float,f[1:])),dtype=np.float32).reshape(-1,2)
        if not np.isfinite(coords).all() or ((coords < -0.0001)|(coords >1.0001)).any():
            raise ValueError(f'Out-of-range normalized polygon: {label_path.name}:{lineno}')
        coords[:,0]*=w; coords[:,1]*=h
        poly=np.rint(coords).astype(np.int32)
        m=np.zeros((h,w),dtype=np.uint8)
        cv2.fillPoly(m,[poly],1)
        if not m.any(): raise ValueError(f'Empty GT mask {label_path.name}:{lineno}')
        masks.append(m)
    return rgb,masks

def decode_coco_rle(rle):
    """Decode the standard COCO compressed RLE with column-major ordering.
    Run-length checks stop execution if the source mask appears corrupted.
    """
    h,w=map(int,rle['size'])
    s=rle['counts']
    if isinstance(s,bytes):s=s.decode('ascii')
    runs=[];p=0
    while p<len(s):
        x=0;k=0
        while True:
            c=ord(s[p])-48;p+=1
            x |= (c&31)<<(5*k)
            k+=1
            if not(c&32):
                if c&16:x |= (-1)<<(5*k)
                break
        if len(runs)>2:x+=runs[-2]
        if x<0:raise ValueError('Invalid negative COCO RLE run')
        runs.append(x)
    if sum(runs)!=h*w:
        raise ValueError(f'Corrupt COCO RLE: run sum {sum(runs)} != mask area {h*w}')
    flattened=np.zeros(h*w, dtype=np.uint8)
    pos=0
    for idx,length in enumerate(runs):
        if idx%2: flattened[pos:pos+length]=1
        pos+=length
    return flattened.reshape((h,w),order='F')

def decode_predictions(pred_list, hw, minimum=BASELINE_CONF):
    result=[]
    for i,d in enumerate(pred_list):
        score=float(d['score'])
        if score<minimum:continue
        rle=dict(d['segmentation'])
        if tuple(rle['size'])!=tuple(hw):
            raise ValueError(f'RLE {rle["size"]} differs from image size {hw}; never resize without a validated transform')
        if isinstance(rle['counts'],str): rle['counts']=rle['counts'].encode('ascii')
        m=decode_coco_rle(rle)
        if m.ndim!=2:raise ValueError('Expected single-instance 2D mask')
        result.append({'index':i,'score':score,'mask':m.astype(np.uint8)})
    return result

def pairwise_iou(gt,pred):
    n,m=len(gt),len(pred)
    mat=np.zeros((n,m), dtype=float)
    for i,g in enumerate(gt):
        for j,p in enumerate(pred):
            inter=np.count_nonzero(g & p['mask'])
            union=np.count_nonzero(g | p['mask'])
            mat[i,j]=inter/union if union else 0.
    return mat

def greedy_match(gt,pred,threshold=MATCH_IOU):
    iou=pairwise_iou(gt,pred)
    choices=sorted([(float(iou[i,j]),i,j) for i in range(len(gt)) for j in range(len(pred)) if iou[i,j]>=threshold],reverse=True)
    used_g,used_p=set(),set(); matches=[]
    for v,i,j in choices:
        if i in used_g or j in used_p: continue
        matches.append((i,j,v));used_g.add(i);used_p.add(j)
    return matches, {'matched_instances':len(matches),'unmatched_gt':len(gt)-len(matches),
                     'unmatched_predictions':len(pred)-len(matches),'mean_matched_iou':np.mean([v for _,_,v in matches]) if matches else np.nan}

# Reference annotation counts must agree with the counts recorded in prior notebooks.
reference_cache={}; audit=[]; removal_audit=[]
for row in selected_cases.itertuples(index=False):
    rgb,gt=load_reference(Path(row.image_path),Path(row.label_path))
    if len(gt)!=row.gt_count: raise AssertionError(f'GT polygon count differs for image ID {row.image_id}: labels={len(gt)}, workbook={row.gt_count}')
    reference_cache[int(row.image_id)]=(rgb,gt)
    for seed in FROZEN:
        baseline=decode_predictions(byseed[seed].get(int(row.image_id),[]),rgb.shape[:2])
        calib=[p for p in baseline if p['score']>=FROZEN[seed]]
        base_match,base_stat=greedy_match(gt,baseline)
        cal_match,cal_stat=greedy_match(gt,calib)
        gtmatched_original=set(j for i,j,v in base_match)
        removed=[j for j,p in enumerate(baseline) if p['score']<FROZEN[seed]]
        removed_matched=sum(j in gtmatched_original for j in removed)
        audit.append({'image_id':row.image_id,'date':row.date,'group':row.group,'seed':seed,'gt_count':len(gt),
                      'baseline_count':len(baseline),'calibrated_count':len(calib),
                      'baseline_matched':base_stat['matched_instances'], 'calibrated_matched':cal_stat['matched_instances'],
                      'baseline_unmatched_gt':base_stat['unmatched_gt'], 'calibrated_unmatched_gt':cal_stat['unmatched_gt'],
                      'baseline_unmatched_predictions':base_stat['unmatched_predictions'],
                      'calibrated_unmatched_predictions':cal_stat['unmatched_predictions'],
                      'removed_detection_count':len(removed),
                      'removed_matched_before_calibration':removed_matched,
                      'removed_unmatched_before_calibration':len(removed)-removed_matched,
                      'baseline_mean_matched_iou':base_stat['mean_matched_iou'],
                      'calibrated_mean_matched_iou':cal_stat['mean_matched_iou']})
        for j in removed:
            p=baseline[j]
            removal_audit.append({'image_id':row.image_id,'seed':seed,'score':p['score'],
                                  'matched_gt_at_baseline_iou50':j in gtmatched_original,
                                  'mask_area_pixels':int(np.count_nonzero(p['mask']))})
matching_df=pd.DataFrame(audit)
removed_df=pd.DataFrame(removal_audit)
print('Pixel masks decoded, polygon GT verified and instance-level descriptive matching completed.')
display(matching_df.groupby('seed')[['removed_detection_count','removed_matched_before_calibration','calibrated_unmatched_gt']].sum())

## 6. Publication-ready visual panels (English)
- **Reference:** annotated ground-truth polygons
- **Original:** masks scoring ≥0.25
- **Calibrated:** masks scoring ≥ the previously frozen validation threshold

The overlays illustrate differences but do not identify root causes by themselves. Full-resolution PNG is exported at 600 dpi; vector PDF/SVG is also saved. All 21 jointly worsened images are included, together with a few comparison cases.

In [ ]:
# Accessible, consistent editorial palette
GT_COLOR=np.array([14,165,130], dtype=np.float32)     # blue-green
ORIGINAL_COLOR=np.array([215,118,24],dtype=np.float32) # amber
CAL_COLOR=np.array([75,90,190],dtype=np.float32)       # indigo

def overlay_masks(image,masks,color,alpha=0.33):
    out=image.astype(np.float32).copy()
    for m in masks:
        mask=m if isinstance(m,np.ndarray) else m['mask']
        where=mask.astype(bool)
        out[where]=(1-alpha)*out[where] + alpha*color
    return np.clip(out,0,255).astype(np.uint8)

def draw_ax(ax,rgb,masks,color,title):
    vis=overlay_masks(rgb,masks,color)
    ax.imshow(vis)
    for item in masks:
        m=item if isinstance(item,np.ndarray) else item['mask']
        cs,_=cv2.findContours(m.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        for poly in cs:
            if len(poly)>=3:
                pts=poly.reshape(-1,2)
                ax.plot(pts[:,0],pts[:,1],linewidth=.85,color=color/255.)
    ax.set_title(title,pad=7)
    ax.axis('off')

def savefig_bundle(fig,stem):
    for ext in ('.png','.pdf','.svg'):
        fig.savefig(FIG_DIR/(stem+ext),dpi=FIG_DPI if ext=='.png' else 250,bbox_inches='tight',pad_inches=.08)
    plt.close(fig)

rendered=[]
for rec in selected_cases.itertuples(index=False):
    rgb,gt=reference_cache[int(rec.image_id)];hw=rgb.shape[:2]
    s=2026
    original=decode_predictions(byseed[s].get(int(rec.image_id),[]),hw)
    cal=[x for x in original if x['score']>=FROZEN[s]]
    fig,axes=plt.subplots(1,3,figsize=(10.2,3.8),layout='constrained')
    draw_ax(axes[0],rgb,gt,GT_COLOR,f'Reference | n = {len(gt)}')
    draw_ax(axes[1],rgb,original,ORIGINAL_COLOR,f'Original | n = {len(original)}, conf ≥ 0.25')
    draw_ax(axes[2],rgb,cal,CAL_COLOR,f'Calibrated | n = {len(cal)}, conf ≥ {FROZEN[s]:.2f}')
    fig.suptitle(f'Image {rec.image_id:03d}  ·  {rec.date}  ·  Seed {s}  ·  {rec.group}',fontsize=12)
    stem=f'Fig30_Image_{int(rec.image_id):03d}_Seed_2026'
    savefig_bundle(fig,stem)
    rendered.append(stem)
print('Exported',len(rendered),'three-panel image figures in PNG (600 dpi), PDF, SVG.')

## 7. Cross-seed audit for the six highest-priority cases
This provides a seven-panel review of the **same image**: reference, original and calibrated predictions for all three seeds. Only the first six critical cases are rendered this way to limit the figure archive size.

In [ ]:
for image_id in critical_ids[:MAX_SEVEN_PANEL_FIGURES]:
    rgb,gt=reference_cache[int(image_id)]
    fig,axes=plt.subplots(2,4,figsize=(13.5,6.8),layout='constrained')
    draw_ax(axes[0,0],rgb,gt,GT_COLOR,f'Reference | {len(gt)} pigs')
    axes[1,0].axis('off')
    axes[1,0].text(.05,.80,f'Image ID {image_id}',transform=axes[1,0].transAxes,weight='bold',fontsize=12)
    axes[1,0].text(.05,.65,f'Date: {str(record.loc[image_id,"fecha"])}',transform=axes[1,0].transAxes)
    axes[1,0].text(.05,.50,'Original confidence = 0.25',transform=axes[1,0].transAxes)
    axes[1,0].text(.05,.36,'Calibrated thresholds from validation',transform=axes[1,0].transAxes)
    for col,s in enumerate(FROZEN,start=1):
        original=decode_predictions(byseed[s].get(int(image_id),[]),rgb.shape[:2])
        calibrated=[p for p in original if p['score']>=FROZEN[s]]
        draw_ax(axes[0,col],rgb,original,ORIGINAL_COLOR,f'Seed {s} | original {len(original)}')
        draw_ax(axes[1,col],rgb,calibrated,CAL_COLOR,f'Seed {s} | calibrated {len(calibrated)}')
    fig.suptitle('Cross-seed comparison of instance masks | Previously inspected test set',fontsize=12)
    stem=f'Fig30_Cross_Seed_{int(image_id):03d}'
    savefig_bundle(fig,stem)
print('Additional seven-panel figures:',min(len(critical_ids),MAX_SEVEN_PANEL_FIGURES))

## 8. Summaries, review sheet, provenance and automatic ZIP export
A manual annotation form is included for expert reviewers (e.g., occlusion, crowding, truncated animals, overlapping masks). These categories remain *unassigned* until a person evaluates the images. Figures and data are explicitly marked descriptive: this is the already-inspected TEST set.

In [ ]:
summary=matching_df.groupby(['seed','group'],as_index=False).agg(
    n_images=('image_id','nunique'),total_gt=('gt_count','sum'),baseline_pred=('baseline_count','sum'),
    calibrated_pred=('calibrated_count','sum'),baseline_matched=('baseline_matched','sum'),
    calibrated_matched=('calibrated_matched','sum'),removed_detections=('removed_detection_count','sum'),
    removed_matched_before=('removed_matched_before_calibration','sum'),
    removed_unmatched_before=('removed_unmatched_before_calibration','sum'))
review=selected_cases[['image_id','date','filename','group','gt_count']].copy()
review['occlusion_observed']=''
review['high_density_observed']=''
review['truncated_animal_observed']=''
review['duplicate_masks_observed']=''
review['missed_animal_observed']=''
review['reviewer_notes']=''
review['reviewer_initials']=''
review['review_date']=''
methods=pd.DataFrame([
    ['Aim','Descriptive visual audit of TEST images previously inspected in experiments 23–29'],
    ['Source masks','Original COCO RLE masks and scores from CERDOS_23_Pred_seed_*.json; no re-inference'],
    ['Baseline','Retain detections with confidence >= 0.25'],
    ['Calibrated','2026: >= 0.70; 2027 and 2028: >= 0.65 (frozen in validation)'],
    ['Instance matching','Greedy one-to-one mask IoU >= 0.50; descriptive only, not COCO evaluation'],
    ['Mask interpretation','Matched/unmatched is a proxy; not an expert-established causal label'],
    ['Scope','21 jointly worsened images + six other residual cases + three improved controls'],
    ['Main limitation','Reused, previously examined test data; no new independent validation'],
    ['GT','Original FaroPigSeg YOLO polygon labels, resolved from train/val/test storage by filename; matched to the experiment 29 counts'],
    ['Experimental partition','Split by date manifest; original FaroPigSeg folder names do not indicate experimental train/val/test'],
    ['Selection','No threshold/model selection from test'],
],columns=['parameter','detail'])
excel=OUT/'CERDOS_30_Analisis_Visual_Errores.xlsx'
with pd.ExcelWriter(excel,engine='openpyxl') as writer:
    summary.to_excel(writer,sheet_name='Summary',index=False)
    selected_cases.to_excel(writer,sheet_name='Selected_Images',index=False)
    critical.to_excel(writer,sheet_name='All_21_Critical',index=False)
    matching_df.to_excel(writer,sheet_name='IoU_Instance_Audit',index=False)
    removed_df.to_excel(writer,sheet_name='Removed_Detections',index=False)
    countcheck.to_excel(writer,sheet_name='Replay_Verification',index=False)
    review.to_excel(writer,sheet_name='Expert_Review_Blank',index=False)
    methods.to_excel(writer,sheet_name='Methods_and_Limits',index=False)
provenance={
    'pipeline':'CERDOS_30_Visual_Error_Audit_v1',
    'runtime':'CPU; precomputed COCO RLE predictions, no new inference or training',
    'evaluation_status':'descriptive; previously inspected test set',
    'source_path_policy':'Lookup by exact filename across FaroPigSeg original train/val/test image and label storage folders; date manifest determines experimental membership',
    'source_storage_folder_counts':{str(k):int(v) for k,v in selected_cases['original_storage_folder'].value_counts().to_dict().items()},
    'confidence_thresholds':{'baseline':BASELINE_CONF,'frozen_validation':FROZEN},
    'mask_iou_matching_threshold':MATCH_IOU,
    'n_test_images_reference':len(priority),
    'n_all_seeds_worsened':len(critical),
    'n_selected_images':len(selected_cases),
    'n_replayed_image_seed_predictions':len(countcheck),
    'replay_all_exact':bool(countcheck['match'].all()),
    'source_23_zip_sha256':hashlib.sha256(ZIP23.read_bytes()).hexdigest(),
    'source_29_excel_sha256':hashlib.sha256(XLSX29.read_bytes()).hexdigest(),
    'split_manifest_sha256':hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
    'protocol_original':protocol['prediction'],
    'python_version':sys.version, 'opencv_version':cv2.__version__,
    'rle_decoder':'COCO compressed RLE; local pure-Python decoder with run-length checksum',
    'pandas_version':pd.__version__, 'numpy_version':np.__version__,
    'limitations':['No external validation','No change to confidence thresholds',
                   'Instance IoU matching is descriptive, not COCO mAP','Expert causal coding not performed']
}
provfile=OUT/'CERDOS_30_Provenance.json'
provfile.write_text(json.dumps(provenance,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
zipout=OUT/'CERDOS_30_Resultados.zip'
with zipfile.ZipFile(zipout,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for file in [excel,provfile]+sorted(FIG_DIR.iterdir()):
        if file.is_file():z.write(file,arcname=str(file.relative_to(OUT)))
print('Excel:',excel)
print('Figures:',len(list(FIG_DIR.glob('*.png'))),'PNG + matching PDF / SVG')
print('ZIP:',zipout, 'Size: %.1f MB'%(zipout.stat().st_size/1e6))
display(summary)
from google.colab import files
files.download(str(zipout))